# Stage 4 Detector -> VLM (Kaggle)

This notebook runs the full Stage 4 pipeline:

`val images -> detector -> pred crops -> Stage3 VLM -> Stage4 eval -> tar.gz`

The final archive is saved to `/kaggle/working`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

DATASET_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/kostyaryazanov/idid-coco-v3'),
    Path('/kaggle/input/idid-coco-v3'),
]
JSONL_REL = Path('stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl')

DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / JSONL_REL).exists():
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        'Could not resolve DATA_ROOT via stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl'
    )

REPO_DIR = Path('/kaggle/working/vlm-for-insulator-defect-detection')
REPO_URL = 'https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git'
RUN_NAME = 'stage4_detector_to_vlm_pred_val_kaggle'

print('DATA_ROOT:', DATA_ROOT)
print('REPO_DIR :', REPO_DIR)
print('RUN_NAME :', RUN_NAME)


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'hydra-core'], check=True)

print('Repo ready:', REPO_DIR)

In [ ]:
import yaml

def pick_existing(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError('Not found among candidates:\\n' + '\\n'.join(str(x) for x in candidates))

# Stage 3 GT labels: resolve exactly like in the working Stage 3 notebooks
gt_jsonl = pick_existing([
    DATA_ROOT / 'stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl',
    DATA_ROOT / 'outputs/stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl',
    REPO_DIR / 'outputs/stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl',
])

# Detector assets are separate from Stage 3 regrouped assets.
# The current Kaggle dataset root may contain only Stage 3 crops/jsonl.
DETECTOR_ASSETS_ROOT_CANDIDATES = [
    Path('/kaggle/input/idid-detector-assets'),
    Path('/kaggle/input/detector-assets'),
    DATA_ROOT,
]

DETECTOR_ASSETS_ROOT = None
for root in DETECTOR_ASSETS_ROOT_CANDIDATES:
    if root.exists():
        DETECTOR_ASSETS_ROOT = root
        break

if DETECTOR_ASSETS_ROOT is None:
    raise FileNotFoundError('Could not resolve DETECTOR_ASSETS_ROOT')

try:
    detector_input_dir = pick_existing([
        DETECTOR_ASSETS_ROOT / 'data/processed/val/images',
        DETECTOR_ASSETS_ROOT / 'val/images',
    ])

    coco_json = pick_existing([
        DETECTOR_ASSETS_ROOT / 'data/processed/val/annotations.json',
        DETECTOR_ASSETS_ROOT / 'val/annotations.json',
    ])

    weights_path = pick_existing([
        DETECTOR_ASSETS_ROOT / 'outputs/train/detector_baseline/best.pth',
        REPO_DIR / 'outputs/train/detector_baseline/best.pth',
    ])
except FileNotFoundError as e:
    raise FileNotFoundError(
        'Notebook 17 is configured for Stage 4 detector->VLM evaluation, but the currently attached Kaggle input appears to contain Stage 3 regrouped assets only. '
        'Resolved DATA_ROOT successfully for stage3_regrouped_v2 JSONL, but could not find detector val images / COCO annotations / weights. '
        'Attach a separate detector-assets Kaggle dataset or export detector assets into a new Kaggle input.\\n\\n'
        f'Original resolution error:\\n{e}'
    )

print('gt_jsonl            :', gt_jsonl)
print('DETECTOR_ASSETS_ROOT:', DETECTOR_ASSETS_ROOT)
print('detector_input_dir  :', detector_input_dir)
print('coco_json           :', coco_json)
print('weights_path        :', weights_path)

ceiling_run_dir = None
for c in [
    DATA_ROOT / 'outputs/stage3_vlm_baseline_runs/stage3_qwen_val_v2_kaggle',
    REPO_DIR / 'outputs/stage3_vlm_baseline_runs/stage3_qwen_val_v2_kaggle',
]:
    if c.exists():
        ceiling_run_dir = c
        break

cfg = {
    'version': 1,
    'name': 'stage4_detector_to_vlm_pred_val_kaggle',
    'stage4': {
        'run_name': RUN_NAME,
        'split': 'val',
        'output_root': str(REPO_DIR / 'outputs/stage4'),
    },
    'detector': {
        'experiment': 'detector_baseline',
        'input_dir': str(detector_input_dir),
        'config_path': str(REPO_DIR / 'src/configs/infer.yaml'),
        'weights_path': str(weights_path),
        'conf_threshold': 0.3,
        'iou_threshold': 0.5,
        'max_detections_per_image': 100,
        'vis_samples': 8,
        'device': 'auto',
    },
    'crop_export': {
        'coco_json': str(coco_json),
        'images_dir': str(detector_input_dir),
        'include_categories': ['insulator_ok', 'defect_flashover', 'defect_broken', 'unknown'],
        'padding_ratio': 0.15,
        'manifest_name': 'pred_manifest.jsonl',
        'summary_name': 'pred_manifest_summary.json',
        'limit': None,
    },
    'vlm': {
        'backend_mode': 'qwen_hf',
        'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'prompt_version': 'qwen_vlm_labels_v1_prompt_v6d_balanced_notaglock',
        'stage3_runner_config': str(REPO_DIR / 'configs/pipeline/stage3_vlm_gt_baseline.yaml'),
        'run_id': 'stage4_pred_vlm',
    },
    'analysis': {
        'ground_truth_jsonl': str(gt_jsonl),
        'match_iou_threshold': 0.5,
        'good_crop_iou_threshold': 0.7,
    },
}

if ceiling_run_dir is not None:
    cfg['analysis']['ceiling_run_dir'] = str(ceiling_run_dir)

cfg_path = REPO_DIR / 'configs/stage4_kaggle_run.yaml'
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding='utf-8')

print('Config path:', cfg_path)
print(cfg_path.read_text(encoding='utf-8'))

In [ ]:
required_paths = [
    REPO_DIR / 'scripts/run_stage4_detector_to_vlm.py',
    detector_input_dir,
    coco_json,
    weights_path,
    gt_jsonl,
]
for p in required_paths:
    if not Path(p).exists():
        raise FileNotFoundError(f'Missing required path: {p}')

print('Preflight OK.')

In [ ]:
os.chdir(REPO_DIR)
cmd = [sys.executable, 'scripts/run_stage4_detector_to_vlm.py', '--config', str(cfg_path)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Stage 4 run finished.')

In [ ]:
run_root = REPO_DIR / 'outputs/stage4' / RUN_NAME
eval_dir = run_root / '04_eval'
metrics_path = eval_dir / 'stage4_metrics.json'
breakdown_path = eval_dir / 'stage4_error_breakdown.json'
summary_path = eval_dir / 'stage4_summary.md'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
breakdown = json.loads(breakdown_path.read_text(encoding='utf-8'))

print('=== Stage 4 rates ===')
for k, v in metrics.get('rates', {}).items():
    print(f'{k}: {v:.4f}' if isinstance(v, (int, float)) else f'{k}: {v}')

print('\n=== Error buckets (counts) ===')
for k, v in breakdown.get('counts', {}).items():
    print(f'{k}: {v}')

print('\nSummary file:', summary_path)

In [ ]:
archive_path = Path('/kaggle/working') / f'stage4_deliverables_{RUN_NAME}.tar.gz'
if archive_path.exists():
    archive_path.unlink()

subprocess.run([
    'tar', '-czf', str(archive_path),
    '-C', '/kaggle/working',
    f'vlm-for-insulator-defect-detection/outputs/stage4/{RUN_NAME}'
], check=True)

size_mb = archive_path.stat().st_size / (1024 * 1024)
print('Archive ready:', archive_path)
print(f'Archive size: {size_mb:.2f} MB')

print('\nImportant files inside run:')
for rel in [
    '01_detector/predictions.json',
    '02_pred_crops/pred_manifest.jsonl',
    '03_vlm_pred/stage4_pred_vlm/predictions_vlm_labels_v1.jsonl',
    '04_eval/stage4_metrics.json',
    '04_eval/stage4_error_breakdown.json',
    '04_eval/stage4_case_table.csv',
    '04_eval/stage4_summary.md',
    '05_compare/ceiling_vs_actual.json',
    'stage4_run_summary.json',
]:
    p = Path('/kaggle/working/vlm-for-insulator-defect-detection/outputs/stage4') / RUN_NAME / rel
    print('-', p, '| exists:', p.exists())

After the run completes, download this file:

`/kaggle/working/stage4_deliverables_stage4_detector_to_vlm_pred_val_kaggle.tar.gz`
